# Ground truth extractive vs SMATCH — Colab Enterprise

Notebook **self-contained** untuk memeriksa setiap dokumen Liputan6 tanpa mengimpor skrip dari repository. Seluruh logika analisis berada di dalam cell notebook, sehingga notebook dapat diunggah sendiri ke Colab Enterprise selama data tersedia pada filesystem runtime.

Untuk setiap dokumen, notebook menghasilkan:

- indeks ground truth dari `extractive_summary`;
- semua pasangan indeks kalimat dengan SMATCH maksimum;
- indikator apakah salah satu/kedua anggota pasangan maksimum merupakan ground truth;
- kalimat dengan weighted degree (jumlah nilai relasi satu baris) tertinggi; dan
- error alignment antara source JSON, arsip AMR, dan adjacency matrix.

## Path data yang digunakan

Semua path input dan output ditulis secara absolut pada cell konfigurasi. Tidak ada path yang diturunkan dari lokasi notebook atau current working directory. Ubah setiap path secara langsung jika data dipindahkan ke Colab Enterprise.

File `.txt` adalah output AMR per kalimat. File `.json` adalah dokumen asli Liputan6; field `extractive_summary` di dalamnya menyediakan indeks ground truth. File AMR `.txt` dan adjacency `.npy` tidak menyimpan ground truth tersebut.

Jika data berada di Cloud Storage (`gs://...`), salin atau mount ke filesystem runtime terlebih dahulu. `pathlib.Path` tidak membaca URI `gs://` secara langsung.


## 1. Dependency opsional

Notebook memerlukan `numpy`, `pandas`, dan `tqdm`. Jika belum tersedia pada kernel, hilangkan komentar pada command berikut dan jalankan satu kali.


In [ ]:
# %pip install --quiet numpy pandas tqdm


## 2. Konfigurasi

Semua path di bawah ini sengaja di-hardcode dan berdiri sendiri. Untuk Windows, pertahankan bentuk raw string (`r'...'`). Saat memakai Colab Enterprise, ganti setiap path Windows dengan path absolut pada runtime, misalnya `/home/jupyter/liputan6/...`.


In [ ]:
from pathlib import Path

SPLIT = 'dev'               # 'train', 'dev', 'test', 'both', atau 'all'
MAX_DOCUMENTS = None       # contoh: 10 untuk sampling; None untuk semua
TIE_TOLERANCE = 1e-8

# Semua lokasi berikut adalah path absolut. Ganti satu per satu di Colab Enterprise.
ADJACENCY_DIRS = {
    'train': Path(r'D:\Github\generate_amr\data\liputan6\adjacency_matrices\train'),
    'dev': Path(r'D:\Github\generate_amr\data\liputan6\adjacency_matrices\dev'),
    'test': Path(r'D:\Github\generate_amr\data\liputan6\adjacency_matrices\test'),
}

AMR_GRAPH_DIRS = {
    'train': Path(r'D:\Github\generate_amr\data\liputan6\amr_graphs\extracted\train'),
    'dev': Path(r'D:\Github\generate_amr\data\liputan6\amr_graphs\extracted\dev'),
    'test': Path(r'D:\Github\generate_amr\data\liputan6\amr_graphs\extracted\test'),
}

SOURCE_DIRS = {
    'train': Path(r'D:\Github\generate_amr\data\liputan6\source\Liputan6\liputan6\liputan6_data\canonical\train'),
    'dev': Path(r'D:\Github\generate_amr\data\liputan6\source\Liputan6\liputan6\liputan6_data\canonical\dev'),
    'test': Path(r'D:\Github\generate_amr\data\liputan6\source\Liputan6\liputan6\liputan6_data\canonical\test'),
}

OUTPUT_PATH = Path(r'D:\Github\generate_amr\data\liputan6\processed\extractive_vs_smatch.csv')

for split_name in ('train', 'dev', 'test'):
    print(f'Adjacency {split_name:5s}:', ADJACENCY_DIRS[split_name])
    print(f'AMR {split_name:5s}      :', AMR_GRAPH_DIRS[split_name])
    print(f'Source {split_name:5s}   :', SOURCE_DIRS[split_name])
print('Output          :', OUTPUT_PATH)


## 3. Preflight data

Cell ini hanya memeriksa dependency dan ketersediaan path. Untuk `SPLIT='both'`, input train dan test harus sama-sama tersedia.


In [ ]:
import importlib.util

required_modules = ['numpy', 'pandas', 'tqdm']
missing_modules = [
    name for name in required_modules if importlib.util.find_spec(name) is None
]
if missing_modules:
    raise ModuleNotFoundError(
        'Dependency belum tersedia: ' + ', '.join(missing_modules)
        + '. Jalankan cell instalasi opsional, lalu ulangi preflight.'
    )

if SPLIT not in {'train', 'dev', 'test', 'both', 'all'}:
    raise ValueError(
        "SPLIT harus 'train', 'dev', 'test', 'both', atau 'all'"
    )
if MAX_DOCUMENTS is not None and MAX_DOCUMENTS < 1:
    raise ValueError('MAX_DOCUMENTS harus None atau integer >= 1')
if TIE_TOLERANCE < 0:
    raise ValueError('TIE_TOLERANCE harus non-negative')

if SPLIT == 'both':
    selected_splits = ['train', 'test']
elif SPLIT == 'all':
    selected_splits = ['train', 'dev', 'test']
else:
    selected_splits = [SPLIT]
required_paths = []
for split_name in selected_splits:
    required_paths.extend([
        (f'adjacency {split_name}', ADJACENCY_DIRS[split_name]),
        (f'AMR .txt {split_name}', AMR_GRAPH_DIRS[split_name]),
        (f'source {split_name}', SOURCE_DIRS[split_name]),
    ])

missing_paths = [(label, path) for label, path in required_paths if not path.exists()]
if missing_paths:
    details = '\n'.join(f'- {label}: {path}' for label, path in missing_paths)
    raise FileNotFoundError('Input berikut belum ditemukan:\n' + details)

for split_name in selected_splits:
    matrix_count = len(list(ADJACENCY_DIRS[split_name].glob('*.npy')))
    source_count = len(list(SOURCE_DIRS[split_name].glob('*.json')))
    print(f'{split_name}: {matrix_count:,} matrices; {source_count:,} source JSON')
print('Preflight OK')


## 4. Fungsi analisis

Bagian ini berisi implementasi lengkap: membaca indeks graph dari nama file `.txt`, memvalidasi alignment, mencari semua edge maksimum, menghitung weighted degree, dan membentuk satu baris hasil per dokumen. Isi graph AMR tidak perlu dibaca ulang karena skor SMATCH sudah tersimpan pada adjacency matrix.


In [ ]:
import csv
import json
import re
from collections import defaultdict

import numpy as np
from tqdm.auto import tqdm

SENTENCE_FILE_RE = re.compile(
    r'^(?P<doc_id>.+)_(?P<sent_idx>\d+)\.txt$'
)

OUTPUT_FIELDS = [
    'split',
    'doc_id',
    'status',
    'error',
    'num_sentences',
    'sentence_indices',
    'extractive_summary_indices',
    'max_smatch_score',
    'max_smatch_pairs',
    'max_smatch_pair_count',
    'max_pair_any_endpoint_in_ground_truth',
    'max_pair_both_endpoints_in_ground_truth',
    'max_weighted_degree',
    'max_weighted_degree_sentence_indices',
    'max_degree_any_sentence_in_ground_truth',
]


def as_json(value):
    return json.dumps(value, ensure_ascii=False, separators=(',', ':'))


def document_sort_key(value):
    return (not value.isdigit(), int(value) if value.isdigit() else value)


def index_amr_directory(amr_dir):
    grouped = defaultdict(list)
    seen = set()
    for graph_path in amr_dir.rglob('*.txt'):
        match = SENTENCE_FILE_RE.fullmatch(graph_path.name)
        if not match:
            continue
        doc_id = match.group('doc_id')
        sent_idx = int(match.group('sent_idx'))
        key = (doc_id, sent_idx)
        if key in seen:
            raise ValueError(
                f'Duplicate sentence {doc_id}_{sent_idx} below {amr_dir}'
            )
        seen.add(key)
        grouped[doc_id].append(sent_idx)
    for indices in grouped.values():
        indices.sort()
    return dict(grouped)


def load_ground_truth(source_path):
    with source_path.open('r', encoding='utf-8') as source_file:
        document = json.load(source_file)
    if 'extractive_summary' not in document:
        raise ValueError('Source JSON has no extractive_summary field')
    if 'clean_article' not in document:
        raise ValueError('Source JSON has no clean_article field')
    indices = document['extractive_summary']
    if not isinstance(indices, list) or not all(
        isinstance(index, int) and not isinstance(index, bool)
        for index in indices
    ):
        raise ValueError('extractive_summary must be a list of integers')
    article_size = len(document['clean_article'])
    invalid = [index for index in indices if index < 0 or index >= article_size]
    if invalid:
        raise ValueError(
            f'Out-of-range ground-truth indices {invalid} for '
            f'{article_size} source sentences'
        )
    return indices, article_size


def validate_adjacency(matrix, expected_size):
    if matrix.ndim != 2 or matrix.shape[0] != matrix.shape[1]:
        raise ValueError(f'Adjacency matrix must be square; found {matrix.shape}')
    if matrix.shape[0] != expected_size:
        raise ValueError(
            f'Matrix has {matrix.shape[0]} rows but AMR directory has '
            f'{expected_size} sentence graphs'
        )
    if not np.isfinite(matrix).all():
        raise ValueError('Adjacency contains NaN or infinite values')
    if not np.allclose(matrix, matrix.T, rtol=1e-5, atol=1e-7):
        raise ValueError('Adjacency matrix is not symmetric')


def strongest_pairs(matrix, sentence_indices, tolerance):
    size = matrix.shape[0]
    if size < 2:
        return None, []
    first_rows, second_rows = np.triu_indices(size, k=1)
    scores = matrix[first_rows, second_rows]
    maximum = float(scores.max())
    tied_positions = np.flatnonzero(
        np.isclose(scores, maximum, rtol=0.0, atol=tolerance)
    )
    pairs = [
        [
            sentence_indices[int(first_rows[position])],
            sentence_indices[int(second_rows[position])],
        ]
        for position in tied_positions
    ]
    return maximum, pairs


def highest_weighted_degree(matrix, sentence_indices, tolerance):
    if matrix.shape[0] == 0:
        return None, []
    degrees = matrix.sum(axis=1, dtype=np.float64)
    maximum = float(degrees.max())
    tied_rows = np.flatnonzero(
        np.isclose(degrees, maximum, rtol=0.0, atol=tolerance)
    )
    return maximum, [sentence_indices[int(row)] for row in tied_rows]


In [ ]:
def analyze_document(
    matrix_path, source_path, sentence_indices, split, tolerance
):
    doc_id = matrix_path.stem
    if len(sentence_indices) != len(set(sentence_indices)):
        raise ValueError('AMR directory contains duplicate sentence indices')

    matrix = np.load(matrix_path, allow_pickle=False)
    validate_adjacency(matrix, len(sentence_indices))
    ground_truth, article_size = load_ground_truth(source_path)

    invalid_archive_indices = [
        index for index in sentence_indices if index < 0 or index >= article_size
    ]
    if invalid_archive_indices:
        raise ValueError(
            'AMR filename indices outside clean_article: '
            + str(invalid_archive_indices)
        )

    maximum_score, pairs = strongest_pairs(
        matrix, sentence_indices, tolerance
    )
    maximum_degree, degree_indices = highest_weighted_degree(
        matrix, sentence_indices, tolerance
    )
    ground_truth_set = set(ground_truth)

    any_endpoint = any(
        first in ground_truth_set or second in ground_truth_set
        for first, second in pairs
    )
    both_endpoints = any(
        first in ground_truth_set and second in ground_truth_set
        for first, second in pairs
    )
    any_degree_sentence = any(
        index in ground_truth_set for index in degree_indices
    )

    return {
        'split': split,
        'doc_id': doc_id,
        'status': 'ok',
        'error': '',
        'num_sentences': len(sentence_indices),
        'sentence_indices': as_json(sentence_indices),
        'extractive_summary_indices': as_json(ground_truth),
        'max_smatch_score': '' if maximum_score is None else maximum_score,
        'max_smatch_pairs': as_json(pairs),
        'max_smatch_pair_count': len(pairs),
        'max_pair_any_endpoint_in_ground_truth': any_endpoint,
        'max_pair_both_endpoints_in_ground_truth': both_endpoints,
        'max_weighted_degree': '' if maximum_degree is None else maximum_degree,
        'max_weighted_degree_sentence_indices': as_json(degree_indices),
        'max_degree_any_sentence_in_ground_truth': any_degree_sentence,
    }


def error_row(split, doc_id, message):
    row = {field: '' for field in OUTPUT_FIELDS}
    row.update({
        'split': split,
        'doc_id': doc_id,
        'status': 'error',
        'error': message,
    })
    return row


def analyze_split(split, max_documents=None):
    matrix_dir = ADJACENCY_DIRS[split]
    amr_dir = AMR_GRAPH_DIRS[split]
    source_dir = SOURCE_DIRS[split]
    amr_index = index_amr_directory(amr_dir)

    matrix_paths = sorted(
        matrix_dir.glob('*.npy'),
        key=lambda path: document_sort_key(path.stem),
    )
    if max_documents is not None:
        matrix_paths = matrix_paths[:max_documents]

    rows = []
    for matrix_path in tqdm(matrix_paths, desc=f'Analyze {split}'):
        doc_id = matrix_path.stem
        try:
            if doc_id not in amr_index:
                raise ValueError('Document is absent from the AMR .txt directory')
            source_path = source_dir / f'{doc_id}.json'
            if not source_path.is_file():
                raise FileNotFoundError(f'Missing source JSON: {source_path}')
            row = analyze_document(
                matrix_path=matrix_path,
                source_path=source_path,
                sentence_indices=amr_index[doc_id],
                split=split,
                tolerance=TIE_TOLERANCE,
            )
        except Exception as exc:
            row = error_row(split, doc_id, str(exc))
        rows.append(row)
    return rows


print('Analysis functions ready')


## 5. Jalankan analisis dan tulis CSV


In [ ]:
rows = []
for split_name in selected_splits:
    rows.extend(analyze_split(split_name, MAX_DOCUMENTS))

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
with OUTPUT_PATH.open('w', encoding='utf-8', newline='') as output_file:
    writer = csv.DictWriter(output_file, fieldnames=OUTPUT_FIELDS)
    writer.writeheader()
    writer.writerows(rows)

valid_count = sum(row['status'] == 'ok' for row in rows)
print(
    f'Checked {len(rows):,} documents: {valid_count:,} OK, '
    f'{len(rows) - valid_count:,} errors'
)
print('CSV:', OUTPUT_PATH)


## 6. Tampilkan hasil dan ringkasan


In [ ]:
import pandas as pd
from IPython.display import display

results = pd.read_csv(OUTPUT_PATH)
valid = results.loc[results['status'].eq('ok')].copy()
errors = results.loc[results['status'].ne('ok')].copy()

print(f'{len(results):,} rows loaded from {OUTPUT_PATH}')
display(results.head(10))


In [ ]:
summary_rows = []
for split_name, frame in valid.groupby('split', sort=False):
    summary_rows.append({
        'split': split_name,
        'documents_ok': len(frame),
        'strongest_pair_touches_ground_truth': int(
            frame['max_pair_any_endpoint_in_ground_truth'].sum()
        ),
        'strongest_pair_both_ground_truth': int(
            frame['max_pair_both_endpoints_in_ground_truth'].sum()
        ),
        'highest_degree_is_ground_truth': int(
            frame['max_degree_any_sentence_in_ground_truth'].sum()
        ),
        'pair_touch_rate': frame[
            'max_pair_any_endpoint_in_ground_truth'
        ].mean(),
        'degree_hit_rate': frame[
            'max_degree_any_sentence_in_ground_truth'
        ].mean(),
    })

summary = pd.DataFrame(summary_rows)
if not summary.empty:
    display(summary.style.format({
        'pair_touch_rate': '{:.1%}',
        'degree_hit_rate': '{:.1%}',
    }))
print(f'Errors: {len(errors):,}')
if not errors.empty:
    display(errors[['split', 'doc_id', 'error']].head(20))


## 7. Inspeksi satu dokumen

Isi `DOCUMENT_ID` untuk memilih dokumen tertentu. Jika `None`, notebook menggunakan baris valid pertama.


In [ ]:
DOCUMENT_ID = None  # contoh: '26428'

if valid.empty:
    print('Tidak ada dokumen berstatus OK untuk diperiksa.')
else:
    candidates = valid if DOCUMENT_ID is None else valid.loc[
        valid['doc_id'].astype(str).eq(str(DOCUMENT_ID))
    ]
    if candidates.empty:
        raise KeyError(f'Document ID {DOCUMENT_ID} tidak ditemukan')
    selected = candidates.iloc[0]
    print('Document ID             :', selected['doc_id'])
    print('Sentence indices        :', json.loads(selected['sentence_indices']))
    print('Ground truth indices    :', json.loads(selected['extractive_summary_indices']))
    print('Maximum SMATCH          :', selected['max_smatch_score'])
    print('Maximum SMATCH pairs    :', json.loads(selected['max_smatch_pairs']))
    print('Any endpoint is GT      :', selected['max_pair_any_endpoint_in_ground_truth'])
    print('Both endpoints are GT   :', selected['max_pair_both_endpoints_in_ground_truth'])
    print('Maximum weighted degree :', selected['max_weighted_degree'])
    print('Highest-degree indices  :', json.loads(selected['max_weighted_degree_sentence_indices']))
    print('Highest degree is GT    :', selected['max_degree_any_sentence_in_ground_truth'])


## 8. Simpan ke Cloud Storage (opsional)

Isi URI tujuan jika CSV perlu dikembalikan ke bucket. Cell tidak melakukan apa pun selama `GCS_OUTPUT_URI` kosong.


In [ ]:
import subprocess

GCS_OUTPUT_URI = ''  # contoh: 'gs://bucket/results/extractive_vs_smatch.csv'

if GCS_OUTPUT_URI:
    upload = subprocess.run(
        ['gcloud', 'storage', 'cp', str(OUTPUT_PATH), GCS_OUTPUT_URI],
        text=True,
        capture_output=True,
    )
    print(upload.stdout)
    if upload.returncode != 0:
        print(upload.stderr)
        raise RuntimeError('Upload ke Cloud Storage gagal')
else:
    print('Output lokal:', OUTPUT_PATH)
